### 주제

학습자의 수료 여부를 예측하는 AI 알고리즘 개발

### 데이터

- **train.csv [파일]**
    - ID : 샘플별 고유 ID
    - generation : BDA 기수
    - school1 : 대학교
    - major type : 복수전공 여부
    - major1_1 : 제1전공
    - major1_2 : 제2전공
    - major_data : 제1전공 전공자 여부
    - 제 1전공에 '컴퓨터', '인공지능', 'AI', '소프트웨어', '데이터', '통계', '산업경영', '산업' 이 포함되면 True
    - job : 현재 직무
    - class1~4 : 수강 분반
    - re_registration : 학기당 새로운 학회원을 모집할 때 재등록 여부
    - contest_award : 공모전 수상 경력
    - nationality : 내/외국인 여부
    - inflow_route : 유입 경로
    - whyBDA : BDA를 선택한 이유
    - what_to_gain : BDA에서 얻고싶은 것
    - hope_for_group : 조별활동 희망 여부
    - previous_class_3~9 : 각 기수를 수강했을 시 분반
    - major_field : 전공 분야
    - desired_career_path : 희망 진로
    - completed_semester : 대학교 이수학기
    - project_type : 팀/개인 중 프로젝트에 참여하고 싶은 형태
    - time_input : 하루에 BDA에 투입 가능한 시간
    - desired_job : 희망 직무
    - certificate_acquisition : 취득한 자격증
    - desired_certificate : 취득을 희망하는 자격증
    - desired_job_except_data : 데이터 외 희망 직무
    - incumbents_level : 어느 정도 연차의 현직자를 원하는지
    - incumbents_lecture : 어떤 주제의 현직자 강의를 원하는지
    - incumbents_company_level : 강연 현직자가 어느정도 규모의 회사를 다니는 사람이었으면 좋겠는지
    - incumbents_lecture_type : 온, 오프라인 중 원하는 현직자 강연 형태
    - incumbents_lecture_scale : 원하는 현직자 강의 규모
    - '3~50명 내외의 강의 리스너와 1명의 현직자' 는 '30~50명 내외의 강의 리스너와 1명의 현직자'를 의미합니다.
    - incumbents_lecture_scale_reason : 현직자 강의 규모 선택 이유
    - interested_company : 관심있는 기업명
    - expected_domain : 희망하는 도메인
    - contest_participation : 데이터 관련 대회 경험
    - idea_contest : 아이디어 공모전에 대한 경험
    - onedayclass_topic : 원데이 클래스 주제
    - **completed : (TARGET) 수료 여부(0 - 미수료 , 1 - 수료)**
- **test.csv [파일]**
    - ID : 샘플별 고유 ID
    - **completed 칼럼 존재하지 않음.**
    - 그 외 train.csv 파일과 구성 동일
    - **sample_submission.csv [파일] - 제출 양식**
    - ID : 샘플별 고유 ID
    - **completed : (TARGET) 수료 여부(0, 1)**

### 코드 흐름

1. 데이터 로드 & 전처리
    - 별도의 복잡한 전처리 X
2. CatBoost
    - categorical 자동 처리
    - encoding 따로 안 해도 됨
3. Optuna 튜닝

In [ ]:
def objective(trial, X, y, groups, cat_features, text_features):
    params = {
        'iterations': trial.suggest_int('iterations', 500, 1500),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'depth': trial.suggest_int('depth', 3, 6),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 5.0, 30.0, log=True),
        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 20, 100),
        'subsample': trial.suggest_float('subsample', 0.5, 0.8),
        'random_strength': trial.suggest_float('random_strength', 1.0, 10.0, log=True),

        # Fixed Params
        'task_type': "GPU",
        'devices': '0',
        'bootstrap_type': "Bernoulli",
        'loss_function': "Logloss",
        'eval_metric': "Logloss",
        'auto_class_weights': "Balanced",
        'random_seed': OPTUNA_SEED,
        'verbose': False,
        'allow_writing_files': False
    }

    scores = []
    seed_everything(OPTUNA_SEED)

    cv = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=OPTUNA_SEED)
    splitter = cv.split(X, y, groups=groups)

4. Cross Validation
    - StratifiedKFold 이용
5. XGBoost

In [ ]:
def objective(trial, X, y, groups):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 500, 2000),
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.1, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'gamma': trial.suggest_float('gamma', 0.0, 5.0),
        'subsample': trial.suggest_float('subsample', 0.5, 0.9),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 0.9),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.01, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.01, 10.0, log=True),
        'scale_pos_weight': trial.suggest_float('scale_pos_weight', 1.5, 4.0),

        'objective': 'binary:logistic',
        'eval_metric': 'logloss',

        'tree_method': 'hist',
        'device': 'cuda',

        'early_stopping_rounds': 50,
        'random_state': OPTUNA_SEED,
        'verbosity': 0
    }

    scores = []
    seed_everything(OPTUNA_SEED)

    cv = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=OPTUNA_SEED)
    splitter = cv.split(X, y, groups=groups)

    for tr_idx, va_idx in splitter:
        X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
        y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]

        model = xgb.XGBClassifier(**params)

        model.fit(
            X_tr, y_tr,
            eval_set=[(X_va, y_va)],
            verbose=False
        )

        pred_proba = model.predict_proba(X_va)[:, 1]
        score_ll = log_loss(y_va, pred_proba)
        scores.append(score_ll)

    return np.mean(scores)

6. 앙상블

In [ ]:
final_pred = (xgb_pred + cat_pred) / 2

7. 제출 파일 생성

### 새롭게 알게 된 내용 / 어려운 내용 / 배울 점

- Optuna를 활용한 하이퍼파라미터 자동 튜닝 방식
    - 기존에는 grid search를 주로 사용했지만 Optuna를 통해 더 효율적으로 최적의 파라미터를 탐색할 수 있다는 점을 알게 되었다.
- StratifiedKFold의 중요성
    - 단순 KFold가 아니라, 클래스 비율을 유지하는 StratifiedKFold를 사용해야 불균형 데이터에서 안정적인 성능 평가가 가능하다는 점을 이해했다.